In [3]:
import os
import shutil
from pathlib import Path
import random
import pandas as pd
from sklearn.model_selection import train_test_split

BASE_DIR = Path("source_tts_dataset")
META_PATH = BASE_DIR / "metadata.csv"
OUT_DIR  = Path("tts_dataset")
VAL_N = 100
TEST_SIZE= 0.09
RNG = 42
random.seed(RNG)

df = pd.read_csv(META_PATH, sep="\t")


val_n = VAL_N
val_df = df.sample(n=val_n, random_state=RNG)
rest_df = df.drop(val_df.index)

train_df, test_df = train_test_split(rest_df, test_size=0.10, random_state=RNG)


def save_split(name, part_df):
    split_dir = OUT_DIR / name
    wavs_dir = split_dir / "wavs"
    wavs_dir.mkdir(parents=True, exist_ok=True)

    meta_out = split_dir / "metadata.csv"
    to_write = part_df[["audio_file", "text"]].copy()
    to_write.to_csv(meta_out, sep="\t", index=False, header=True, encoding="utf-8")

    copied, failed = 0, 0
    for _, r in part_df.iterrows():
        src = BASE_DIR / "wavs" / r["audio_file"]
        dst = wavs_dir / Path(r["audio_file"]).name
        try:
            shutil.copy2(src, dst)
            copied += 1
        except Exception as e:
            failed += 1
            print(f"Не удалось скопировать {src} -> {dst}: {e}")
    return copied, failed

cop_train = save_split("train", train_df)
cop_val   = save_split("val",   val_df)
cop_test  = save_split("test",  test_df)


print(f"Всего валидных примеров: {len(df)}")
print(f"val:   {len(val_df)} (скопировано {cop_val[0]}, ошибок {cop_val[1]})")
print(f"test:  {len(test_df)} (скопировано {cop_test[0]}, ошибок {cop_test[1]})")
print(f"train: {len(train_df)} (скопировано {cop_train[0]}, ошибок {cop_train[1]})")
print(f"\nСохранено в: {OUT_DIR}/{{train,val,test}}")

Всего валидных примеров: 13805
val:   100 (скопировано 100, ошибок 0)
test:  1371 (скопировано 1371, ошибок 0)
train: 12334 (скопировано 12334, ошибок 0)

Сохранено в: tts_dataset/{train,val,test}
